# Classifier Matrix Orchestrator

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

## Orchestratore matrice classificatori

Costruisce la matrice, mostra conteggi, e puo' avviare/mettere in pausa lo scheduler. **Non esegue mai il locked test**: quella cella e' assente per costruzione, non solo disabilitata — la conferma `--confirm-locked-test` vive esclusivamente in `scripts/finalize_locked_test_stage.py`, eseguito da riga di comando.

### 1. Costruire/ricostruire la matrice (non distruttivo: lo stato e' ricostruito dagli artefatti)

In [1]:
from pathlib import Path
import json, sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_generator_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import build_classifier_experiment_matrix as build_matrix
import resume_classifier_experiment_matrix as resume_matrix
import status_classifier_experiment_matrix as status_mod

STAGE = 1  # 1 = screening completo; 2 = solo dopo SELECTED_GENERATOR_UNION firmato

payload = build_matrix.build_and_write(PROJECT_ROOT, STAGE)
stage_jobs = [j for j in payload["jobs"] if j["stage"] == STAGE]
print(f"Stage {STAGE}: {len(stage_jobs)} job")


Stage 1: 336 job


### 2. Conteggi per architettura / regime / profilo risorsa

In [2]:

report = status_mod.build_report(PROJECT_ROOT, stage=STAGE)
status_mod.print_report(report)


total jobs: 336
by status: {"PENDING": 336}
by architecture: {"resnet50": 84, "maxvit512": 84, "mammofm": 84, "raddino": 84}
by resource profile: {"medium": 252, "heavy": 84}


### 3. Avviare lo scheduler (dry-run di default: imposta DRY_RUN=False solo da riga di comando fuori da questo notebook)

In [3]:

import run_classifier_experiment_matrix as run_matrix
from classifier_gpu_scheduler import query_gpus_live

DRY_RUN = True  # questo notebook non avvia mai training reali; per il vero avvio usa la CLI
TARGET_5060_JOBS = 3
TARGET_3060_JOBS = 2

gpus = query_gpus_live()
result = run_matrix.run(PROJECT_ROOT, STAGE, "auto", TARGET_5060_JOBS, TARGET_3060_JOBS, dry_run=DRY_RUN, gpus=gpus)
print(f"GPU rilevate: {result['gpus']}")
admitted = [p for p in result["plan"] if p["admitted"]]
print(f"job ammessi in questo piano: {len(admitted)} / {len(result['plan'])}")
for p in admitted[:10]:
    print(" ", p)


GPU rilevate: ['NVIDIA GeForce RTX 3060', 'NVIDIA GeForce RTX 5060 Ti']
job ammessi in questo piano: 2 / 336
  {'experiment_id': 'resnet50__R__seed17', 'admitted': True, 'gpu_key': 'rtx_5060_ti_16gb', 'gpu_uuid': 'GPU-82ec33a5-8b4f-40d0-ead7-8d1d9679055d', 'estimated_peak_mb': 6000.0}
  {'experiment_id': 'resnet50__R__seed42', 'admitted': True, 'gpu_key': 'rtx_3060_12gb', 'gpu_uuid': 'GPU-9a95e5e1-599d-6315-52fb-b6a602960eb8', 'estimated_peak_mb': 6000.0}


### 4. Riprendere dopo un crash/riavvio

In [4]:

resume_result = resume_matrix.resume(PROJECT_ROOT, stage=STAGE)
print(f"scansionati {resume_result['total_scanned']} job, {len(resume_result['changed'])} cambi di stato")


scansionati 336 job, 0 cambi di stato


### 5. Finalizzare validation / preparare il lock (mai eseguito automaticamente da questa cella)

In [5]:

print("Per finalizzare la validation Stage 1 (calcola SELECTED_GENERATOR_UNION):")
print("  python scripts/finalize_validation_stage.py --stage 1")
print("Per preparare (non bloccare) i pannelli Stage 2:")
print("  python scripts/finalize_validation_stage.py --stage 2")
print("Il locked test richiede sempre un comando esplicito separato, mai una cella notebook:")
print("  python scripts/finalize_locked_test_stage.py --confirm-locked-test")


Per finalizzare la validation Stage 1 (calcola SELECTED_GENERATOR_UNION):
  python scripts/finalize_validation_stage.py --stage 1
Per preparare (non bloccare) i pannelli Stage 2:
  python scripts/finalize_validation_stage.py --stage 2
Il locked test richiede sempre un comando esplicito separato, mai una cella notebook:
  python scripts/finalize_locked_test_stage.py --confirm-locked-test
